### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [65]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [66]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [67]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np
import pandas as pd

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [68]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [69]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [70]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [71]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [72]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [73]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

Es muy útil tener el diccionario opuesto que va de índices a términos

In [74]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [75]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [76]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [77]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [78]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [79]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [80]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [81]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [82]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [83]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [84]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [85]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [86]:
f1_score(y_test, y_pred, average='macro')

np.float64(0.5854345727938506)

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [87]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import ParameterGrid

# Para reproducibilidad
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# --------------------------------------------------
# 0) CARGA DE DATOS
# --------------------------------------------------
newsgroups_train = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
)

newsgroups_test = fetch_20newsgroups(
    subset='test',
    remove=('headers', 'footers', 'quotes')
)

y_train = newsgroups_train.target
y_test = newsgroups_test.target
target_names = newsgroups_train.target_names

print(f"Cantidad de documentos de entrenamiento: {len(newsgroups_train.data)}")
print(f"Cantidad de documentos de test: {len(newsgroups_test.data)}")
print(f"Cantidad de clases: {len(target_names)}")
print(target_names)

Cantidad de documentos de entrenamiento: 11314
Cantidad de documentos de test: 7532
Cantidad de clases: 20
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


## Ejercicio 1: Vectorización de los documentos

In [88]:
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)
X_test = tfidfvect.transform(newsgroups_test.data)

idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_test: {X_test.shape}")
print(f"Tamaño del vocabulario: {len(tfidfvect.vocabulary_)}")

Shape de X_train: (11314, 101631)
Shape de X_test: (7532, 101631)
Tamaño del vocabulario: 101631


In [89]:
# Tomo 5 documentos al azar y busco los 5 más similares
random_docs = rng.choice(X_train.shape[0], size=5, replace=False)

similarity_results = []

for doc_idx in random_docs:
    cossim = cosine_similarity(X_train[doc_idx], X_train)[0]
    most_similar = np.argsort(cossim)[::-1][1:6]  # excluimos el mismo documento
    
    print("=" * 100)
    print(f"DOCUMENTO BASE - índice: {doc_idx}")
    print(f"Clase real: {target_names[y_train[doc_idx]]}")
    print("Texto base, primeros 700 caracteres:")
    print(newsgroups_train.data[doc_idx][:700].replace("\n", " "))
    print("\nTop 5 documentos más similares:")
    
    for rank, sim_idx in enumerate(most_similar, start=1):
        base_class = target_names[y_train[doc_idx]]
        sim_class = target_names[y_train[sim_idx]]
        same_class = base_class == sim_class
        
        similarity_results.append({
            "documento_base": doc_idx,
            "clase_base": base_class,
            "rank": rank,
            "documento_similar": sim_idx,
            "clase_similar": sim_class,
            "cosine_similarity": cossim[sim_idx],
            "misma_clase": same_class
        })
        
        print(f"\n{rank}) índice: {sim_idx}")
        print(f"   Similaridad coseno: {cossim[sim_idx]:.4f}")
        print(f"   Clase: {sim_class}")
        print(f"   ¿Misma clase?: {same_class}")
        print(f"   Texto, primeros 350 caracteres: {newsgroups_train.data[sim_idx][:350].replace(chr(10), ' ')}")


DOCUMENTO BASE - índice: 8754
Clase real: talk.religion.misc
Texto base, primeros 700 caracteres:
 /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your business.    Some people may also reason that by reading the bible and being a Xtian you are permanently damaging your brain.  By your logic, it would be OK for them to come into your home, take away your bible, and send you off to "re-education camps" to save your mind from ruin.  Are you ready for that?        /(hudson) /And why is there nothing wrong with it?  Because you say so?  Who gave you /the authority to say that, and set the standard for morality?  Why?  Because:  I am a living, thinkin

Top 5 documentos más similares:

1) índice: 6552
   Similaridad coseno: 0.4904
   Clase: talk.religion.misc
   ¿Misma clase?: True
   Texto, primeros 350 caracteres:  If I have a habit that I really w

### Justificación Ej 1

En general, los documentos más similares tienden a pertenecer a la misma clase y, si no lo son, a clases relativamente parecidas. Esto ocurre porque TF-IDF asigna mayor importancia a términos representativos del texto. Por ejemplo, documentos sobre política y religión tienen términos cercanos.

Por este motivo, si bien lo más probable es que detecte textos de la misma clase también puede llegar a "confundirse" con un documento de una clase relativamente cercana.

## Ejercicio 2: Construcción de modelo de clasificación por prototipos (tipo zero-shot)

In [90]:
similarities_test_train = cosine_similarity(X_test, X_train)

# Para cada documento de test, buscamos el documento de train más similar
nearest_train_idx = np.argmax(similarities_test_train, axis=1)

y_pred_proto = y_train[nearest_train_idx] #El indice nearest_train_idx nos dice que columna (train) esta el mas similar al texto de test (fila)

proto_accuracy = accuracy_score(y_test, y_pred_proto)
proto_f1_macro = f1_score(y_test, y_pred_proto, average='macro')

print(f"Accuracy del clasificador por prototipos: {proto_accuracy:.4f}")
print(f"F1 macro del clasificador por prototipos: {proto_f1_macro:.4f}")

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_proto, target_names=target_names))


Accuracy del clasificador por prototipos: 0.5089
F1 macro del clasificador por prototipos: 0.5050

Reporte de clasificación:
                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.ele

### Justificación Ej 2
Se puede observar que no da buenos resultados para la mayoría de los textos, esto se debe que al ser una clasificación basada en memoria: no aprende parámetros, sino que asigna la clase del documento de entrenamiento más parecido. Su desempeño depende de que exista en el set de entrenamiento un texto muy similar en el set de test (misma temática y aún mejor, mismo vocabulario).

## Ejercicio 3: Clasificación Naïve Bayes

In [91]:
experiments = []

vectorizer_grid = list(ParameterGrid({
    "max_df": [0.7, 0.85, 0.95],
    "min_df": [1, 2, 5],
    "max_features": [None, 30000, 50000],
    "stop_words": [None, "english"],
    "sublinear_tf": [False, True],
    "use_idf": [True],
    "norm": ["l2"]
}))

model_configs = []

for alpha in [0.01, 0.05, 0.1, 0.5, 1.0]:
    model_configs.append(("MultinomialNB", MultinomialNB(alpha=alpha), {"alpha": alpha}))
    model_configs.append(("ComplementNB", ComplementNB(alpha=alpha), {"alpha": alpha}))

for vect_params in vectorizer_grid:
    vect = TfidfVectorizer(**vect_params)
    
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    
    for model_name, model, model_params in model_configs:
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        
        f1_macro = f1_score(y_test, pred, average="macro")
        acc = accuracy_score(y_test, pred)
        
        experiments.append({
            "model": model_name,
            "f1_macro": f1_macro,
            "accuracy": acc,
            **model_params,
            **vect_params,
            "vocab_size": Xtr.shape[1]
        })

results_df = pd.DataFrame(experiments).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df.head(15)

,model,f1_macro,accuracy,alpha,max_df,max_features,min_df,norm,stop_words,sublinear_tf,use_idf,vocab_size
0,ComplementNB,0.698038,0.715481,0.5,0.95,NaN,2,l2,None,False,True,39423
1,ComplementNB,0.698038,0.715481,0.5,0.85,50000.0,2,l2,None,False,True,39423
2,ComplementNB,0.698038,0.715481,0.5,0.85,NaN,2,l2,None,False,True,39423
3,ComplementNB,0.698038,0.715481,0.5,0.95,50000.0,2,l2,None,False,True,39423
4,ComplementNB,0.697877,0.715746,0.5,0.70,50000.0,1,l2,None,False,True,50000
5,ComplementNB,0.697866,0.715348,0.5,0.70,NaN,2,l2,None,False,True,39421
6,ComplementNB,0.697866,0.715348,0.5,0.70,50000.0,2,l2,None,False,True,39421
7,ComplementNB,0.697805,0.716543,0.5,0.70,NaN,1,l2,english,False,True,101322
8,ComplementNB,0.697805,0.716543,0.5,0.85,NaN,1,l2,english,False,True,101322
9,ComplementNB,0.697805,0.716543,0.5,0.95,NaN,1,l2,english,False,True,101322


In [92]:
#Mejores resultados
best_result = results_df.iloc[0]
best_result

model           ComplementNB
f1_macro            0.698038
accuracy            0.715481
alpha                    0.5
max_df                  0.95
max_features             NaN
min_df                     2
norm                      l2
stop_words              None
sublinear_tf           False
use_idf                 True
vocab_size             39423
Name: 0, dtype: object

### Justificación ejercicio 3

Los modelos Naïve Bayes superan notablemente al clasificador por prototipos porque aprenden una distribución global de términos por clase. En vez de depender de un único documento parecido, estiman qué términos son más característicos de cada categoría.

"ComplementNB" suele obtener muy buen rendimiento en clasificación de texto porque corrige algunas limitaciones de "MultinomialNB", especialmente cuando hay clases con distribuciones no perfectamente balanceadas.

## Transponer la matriz documento-término.

In [97]:
# Usamos el vectorizador del ejercicio 1
X_terms = X_train.T
feature_names = np.array(tfidfvect.get_feature_names_out())

manual_words = ["money", "car", "house", "tree", "soccer"]

word_similarity_results = []

for word in manual_words:
    if word not in tfidfvect.vocabulary_:
        print(f"La palabra '{word}' no está en el vocabulario.")
        continue
    
    word_idx = tfidfvect.vocabulary_[word]
    cossim_words = cosine_similarity(X_terms[word_idx], X_terms)[0]
    most_similar_word_idx = np.argsort(cossim_words)[::-1][1:6]
    
    print("=" * 80)
    print(f"Palabra base: {word}")
    print("Top 5 palabras más similares:")
    
    for rank, sim_word_idx in enumerate(most_similar_word_idx, start=1):
        similar_word = feature_names[sim_word_idx]
        score = cossim_words[sim_word_idx]
        
        word_similarity_results.append({
            "palabra_base": word,
            "rank": rank,
            "palabra_similar": similar_word,
            "cosine_similarity": score
        })
        
        print(f"{rank}) {similar_word:20s} similarity={score:.4f}")


Palabra base: money
Top 5 palabras más similares:
1) buyback              similarity=0.2041
2) computerize          similarity=0.2041
3) fundraising          similarity=0.1688
4) to                   similarity=0.1640
5) spend                similarity=0.1616
Palabra base: car
Top 5 palabras más similares:
1) cars                 similarity=0.1797
2) criterium            similarity=0.1770
3) civic                similarity=0.1748
4) owner                similarity=0.1689
5) dealer               similarity=0.1681
Palabra base: house
Top 5 palabras más similares:
1) white                similarity=0.2501
2) senate               similarity=0.2484
3) cpr                  similarity=0.2372
4) miyazawa             similarity=0.2211
5) deposit              similarity=0.2015
Palabra base: tree
Top 5 palabras más similares:
1) decorations          similarity=0.4254
2) flourish             similarity=0.3873
3) righteous            similarity=0.2628
4) withering            similarity=0.2586
5) ed

### Justificación ejercicio 4

La similaridad entre palabras no se calcula por significado lingüístico directo, sino por ladistribución de aparición en documentos. Es decir, dos palabras aparecen como similares si suelen estar presentes en documentos parecidos.